In [1]:
# ============================
# 1: Install solvers
# ============================




import pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Binary,
    Objective, Constraint, maximize, SolverFactory, value
)

solver_name = None
for cand in ["appsi_highs", "highs", "cbc", "glpk"]:
    try:
        s = SolverFactory(cand)
        if s is not None and s.available(False):
            solver_name = cand
            break
    except Exception:
        continue

if solver_name is None:
    raise RuntimeError("No MILP solver found")

print(f"Using solver: {solver_name}")
solver = SolverFactory(solver_name)

# =========
# 2. LOAD DATA
# =========

excel_file = "rangers_lineup_data_new.xlsx"
sheet_name = "Rangers Stats"

df = pd.read_excel(excel_file, sheet_name=sheet_name)

df["Primary Pos"] = df["Primary Pos"].astype(str).str.strip()
df["Secondary Pos"] = df["Secondary Pos"].astype(str).str.strip()
df["Name"] = df["Name"].astype(str).str.strip()
df["Bats"] = df["Bats"].astype(str).str.strip()

df["wOBA"] = df["wOBA"].astype(float)
df["wOBA_R"] = df["wOBA_vs_RHP"].astype(float)
df["wOBA_L"] = df["wOBA_vs_LHP"].astype(float)

df["BB%"] = df["BB%"].astype(str).str.replace('%', '').astype(float) / 100

players = df["Name"].tolist()

# Allow DH as a position so bat-only guys can play
F = ["C", "1B", "2B", "3B", "SS", "LF", "CF", "RF", "DH"]

Using solver: appsi_highs


In [3]:
from pyomo.environ import *

# =========
# PARAMETERS
# =========

q = {}
for _, row in df.iterrows():
    name = row["Name"]
    overall = row["wOBA"]
    wR = row["wOBA_vs_RHP"]
    wL = row["wOBA_vs_LHP"]


    q_R = 0.5 * wR + 0.5 * overall
    q_L = 0.5 * wL + 0.5 * overall


    q[(name, "R")] = q_R
    q[(name, "L")] = q_L


# Lineup weights
raw_w = {
    1: 1.00, 2: 0.97, 3: 0.94, 4: 0.91, 5: 0.88,
    6: 0.85, 7: 0.82, 8: 0.79, 9: 0.76
}
total_raw = sum(raw_w.values())
w = {s: raw_w[s]/total_raw for s in raw_w}

# =========
# DEFENSE (position weighted)
# =========

max_def = df["Def/162"].abs().max()
def_score = {row["Name"]: row["Def/162"] / max_def for _, row in df.iterrows()}

pos_def_weight = {
    "C": 1.35,
    "SS": 1.30,
    "CF": 1.20,
    "2B": 1.10,
    "3B": 1.05,
    "1B": 1.00,
    "LF": 1.00,
    "RF": 1.00,
    "DH": 0.00
}



# =========
# ELIGIBILITY + POSITION TYPE MAP
# =========

eligible = {}
pos_type = {}

for _, row in df.iterrows():
    name = row["Name"]
    primary = row["Primary Pos"]
    secondary = row["Secondary Pos"]

    pos_type[name] = (primary, secondary)

    for f in F:
        eligible[(name, f)] = 0

    if primary in F:
        eligible[(name, primary)] = 1
    if isinstance(secondary, str) and secondary in F:
        eligible[(name, secondary)] = 1

# =========
# METRICS
# =========

onbase_score = {}
power_score = {}

for _, row in df.iterrows():
    name = row["Name"]
    onbase_score[name] = 0.7 * row["OBP"] + 0.3 * row["BB%"]
    power_score[name] = 0.7 * row["SLG"] + 0.3 * row["ISO"]

top_onbase = sorted(players, key=lambda x: onbase_score[x], reverse=True)[:5]
top_power = sorted(players, key=lambda x: power_score[x], reverse=True)[:5]
valid_pairs = [(i, j) for i in top_onbase for j in top_power if i != j]

def next_spots(s):
    return [((s) % 9) + 1, ((s+1) % 9) + 1]

# =========
# MODEL
# =========

def solve_lineup(k_hand):

    model = ConcreteModel()

    model.I = Set(initialize=players)
    model.S = Set(initialize=list(range(1,10)))
    model.F = Set(initialize=F)
    model.Pairs = Set(initialize=valid_pairs, dimen=2)

    model.q = Param(model.I, initialize=lambda m,i: q[(i,k_hand)])
    model.w = Param(model.S, initialize=lambda m,s: w[s])
    model.eligible = Param(model.I, model.F, initialize=lambda m,i,f: eligible[(i,f)])

    model.x = Var(model.I, model.S, within=Binary)
    model.y = Var(model.I, model.F, within=Binary)
    model.z = Var(model.Pairs, model.S, within=Binary)

    alpha = 0.05 #Weight of Onbase vs Power factor
    sec_penalty = 0.02 #Secondary Pos Penalty
    def_alpha = 0.05  #Weight of Defense

    # =========
    # OBJECTIVE
    # =========

    def obj_rule(m):

        # offense
        base = sum(
            m.w[s] * m.q[i] * m.x[i,s]
            for i in m.I for s in m.S
        )

        # synergy
        synergy = sum(
            alpha * onbase_score[i] * power_score[j] * m.z[i,j,s]
            for (i,j) in m.Pairs
            for s in m.S
        )

        # defense (position-weighted)
        defense = sum(
            def_alpha *
            def_score[i] *
            pos_def_weight[f] *
            m.y[i,f]
            for i in m.I for f in m.F if f != "DH"
        )

        # secondary position penalty
        penalty = sum(
            sec_penalty *
            m.y[i,f]
            for i in m.I for f in m.F
            if f == pos_type[i][1]   # secondary position
        )

        return base + synergy + defense - penalty

    model.Obj = Objective(rule=obj_rule, sense=maximize)

    # =========
    # CONSTRAINTS
    # =========

    def lineup_complete(m,s):
        return sum(m.x[i,s] for i in m.I) == 1
    model.LineupComplete = Constraint(model.S, rule=lineup_complete)

    def one_spot(m,i):
        return sum(m.x[i,s] for s in m.S) <= 1
    model.OneSpot = Constraint(model.I, rule=one_spot)

    def one_pos(m,f):
        if f == "DH":
            return sum(m.y[i,f] for i in m.I) <= 1
        return sum(m.y[i,f] for i in m.I) == 1
    model.OnePos = Constraint(model.F, rule=one_pos)

    def one_pos_player(m,i):
        return sum(m.y[i,f] for f in m.F) <= 1
    model.OnePosPlayer = Constraint(model.I, rule=one_pos_player)

    def eligibility_rule(m,i,f):
        return m.y[i,f] <= m.eligible[i,f]
    model.Elig = Constraint(model.I, model.F, rule=eligibility_rule)

    def link(m,i):
        return sum(m.x[i,s] for s in m.S) == sum(m.y[i,f] for f in m.F)
    model.Link = Constraint(model.I, rule=link)

    # =========
    # SYNERGY
    # =========

    def z1(m,i,j,s):
        return m.z[i,j,s] <= m.x[i,s]
    model.Z1 = Constraint(model.Pairs, model.S, rule=z1)

    def z2(m,i,j,s):
        return m.z[i,j,s] <= sum(m.x[j,t] for t in next_spots(s))
    model.Z2 = Constraint(model.Pairs, model.S, rule=z2)

    def z3(m,i,j,s):
        return m.z[i,j,s] >= m.x[i,s] + sum(m.x[j,t] for t in next_spots(s)) - 1
    model.Z3 = Constraint(model.Pairs, model.S, rule=z3)

    # =========
    # SOLVE
    # =========

    solver = SolverFactory("highs")
    solver.options["time_limit"] = 60
    solver.options["mip_rel_gap"] = 0.05

    results = solver.solve(model, tee=True)

    print(f"\nTermination vs {k_hand}HP:", results.solver.termination_condition)

    print(f"\nOptimal lineup vs {k_hand}HP:")
    lineup = []
    for s in model.S:
        for i in model.I:
            if value(model.x[i,s]) > 0.5:
                lineup.append((s,i))

    lineup.sort()

    for s,i in lineup:
        pos = [f for f in model.F if value(model.y[i,f]) > 0.5]
        print(f"  Spot {s}: {i} ({pos[0] if pos else 'DH'})")

    print("Objective value:", value(model.Obj))

    return model


# =========
# RUN
# =========

model_r = solve_lineup("R")
model_l = solve_lineup("L")




Termination vs RHP: optimal

Optimal lineup vs RHP:
  Spot 1: Corey Seager (SS)
  Spot 2: Mitch Garver (DH)
  Spot 3: Adolis García (RF)
  Spot 4: Rowdy Tellez (1B)
  Spot 5: Marcus Semien (2B)
  Spot 6: Josh Jung (3B)
  Spot 7: Jonah Heim (C)
  Spot 8: Leody Taveras (CF)
  Spot 9: Wyatt Langford (LF)
Objective value: 0.45865399582619254

Termination vs LHP: optimal

Optimal lineup vs LHP:
  Spot 1: Mitch Garver (DH)
  Spot 2: Adolis García (RF)
  Spot 3: Carson Kelly (C)
  Spot 4: Josh Jung (3B)
  Spot 5: Leody Taveras (CF)
  Spot 6: Marcus Semien (2B)
  Spot 7: Wyatt Langford (LF)
  Spot 8: Nathaniel Lowe (1B)
  Spot 9: Corey Seager (SS)
Objective value: 0.46188495657154355


In [4]:
from pyomo.environ import value

def breakdown_contributions(model, k_hand):

    print(f"\n--- Contribution Breakdown vs {k_hand}HP ---\n")

    total_base = 0
    total_def = 0
    total_penalty = 0

    for i in model.I:
        # check if player is selected
        x_val = sum(value(model.x[i,s]) for s in model.S)
        if x_val < 0.5:
            continue

        # find batting spot
        s = next(s for s in model.S if value(model.x[i,s]) > 0.5)

        # find position
        f = next(f for f in model.F if value(model.y[i,f]) > 0.5)

        # offense contribution
        base = value(model.w[s]) * value(model.q[i])

        # defense contribution (exclude DH)
        if f != "DH":
            defense = def_alpha * def_score[i] * pos_def_weight[f]
        else:
            defense = 0

        # secondary penalty (excluding DH)
        if f == pos_type[i][1] and f != "DH":
            penalty = sec_penalty
        else:
            penalty = 0

        total_base += base
        total_def += defense
        total_penalty += penalty

        print(f"{i:20s} | Spot {s} | {f:2s} | "
              f"Base: {base:.4f} | Def: {defense:.4f} | Penalty: {penalty:.4f}")

    # --- SYNERGY ---
    total_syn = 0
    for (i,j) in model.Pairs:
        for s in model.S:
            if value(model.z[i,j,s]) > 0.5:
                syn = alpha * onbase_score[i] * power_score[j]
                total_syn += syn
                print(f"SYNERGY: {i} -> {j} at spot {s} = {syn:.4f}")

    print("\n--- Totals ---")
    print(f"Base:     {total_base:.4f}")
    print(f"Defense:  {total_def:.4f}")
    print(f"Penalty: -{total_penalty:.4f}")
    print(f"Synergy:  {total_syn:.4f}")
    print(f"TOTAL:    {total_base + total_def - total_penalty + total_syn:.4f}")

In [11]:
alpha = 0.05 #Weight of Onbase vs Power factor
sec_penalty = 0.02 #Secondary Pos Penalty
def_alpha = 0.05  #Weight of Defense
breakdown_contributions(model_r, "R")
#breakdown_contributions(model_l, "L")


--- Contribution Breakdown vs RHP ---

Corey Seager         | Spot 1 | SS | Base: 0.0501 | Def: 0.0290 | Penalty: 0.0000
Marcus Semien        | Spot 5 | 2B | Base: 0.0356 | Def: 0.0454 | Penalty: 0.0000
Wyatt Langford       | Spot 9 | LF | Base: 0.0309 | Def: 0.0117 | Penalty: 0.0000
Adolis García        | Spot 3 | RF | Base: 0.0371 | Def: -0.0199 | Penalty: 0.0000
Josh Jung            | Spot 6 | 3B | Base: 0.0332 | Def: 0.0131 | Penalty: 0.0000
Leody Taveras        | Spot 8 | CF | Base: 0.0301 | Def: 0.0128 | Penalty: 0.0000
Jonah Heim           | Spot 7 | C  | Base: 0.0294 | Def: 0.0392 | Penalty: 0.0000
Mitch Garver         | Spot 2 | DH | Base: 0.0449 | Def: 0.0000 | Penalty: 0.0000
Rowdy Tellez         | Spot 4 | 1B | Base: 0.0388 | Def: -0.0171 | Penalty: 0.0000
SYNERGY: Mitch Garver -> Rowdy Tellez at spot 2 = 0.0056
SYNERGY: Mitch Garver -> Adolis García at spot 2 = 0.0054
SYNERGY: Corey Seager -> Mitch Garver at spot 1 = 0.0061
SYNERGY: Corey Seager -> Adolis García at spot 1

In [8]:
breakdown_contributions(model_l, "L")


--- Contribution Breakdown vs LHP ---

Corey Seager         | Spot 9 | SS | Base: 0.0342 | Def: 0.0290 | Penalty: 0.0000
Marcus Semien        | Spot 6 | 2B | Base: 0.0348 | Def: 0.0454 | Penalty: 0.0000
Wyatt Langford       | Spot 7 | LF | Base: 0.0361 | Def: 0.0117 | Penalty: 0.0000
Nathaniel Lowe       | Spot 8 | 1B | Base: 0.0326 | Def: -0.0220 | Penalty: 0.0000
Adolis García        | Spot 2 | RF | Base: 0.0384 | Def: -0.0199 | Penalty: 0.0000
Josh Jung            | Spot 4 | 3B | Base: 0.0381 | Def: 0.0131 | Penalty: 0.0000
Leody Taveras        | Spot 5 | CF | Base: 0.0312 | Def: 0.0128 | Penalty: 0.0000
Mitch Garver         | Spot 1 | DH | Base: 0.0501 | Def: 0.0000 | Penalty: 0.0000
Carson Kelly         | Spot 3 | C  | Base: 0.0501 | Def: 0.0302 | Penalty: 0.0000
SYNERGY: Mitch Garver -> Adolis García at spot 1 = 0.0054
SYNERGY: Corey Seager -> Mitch Garver at spot 9 = 0.0061
SYNERGY: Corey Seager -> Adolis García at spot 9 = 0.0053
SYNERGY: Nathaniel Lowe -> Corey Seager at spot